# 03 Wikipedia Scraping

**Phase 4 scope note:** This notebook documents the Wikipedia city metadata
scraping policy for Phase 4 of the euro-air-quality-pipeline project.
It covers source access, raw-data hygiene, naming conventions,
rate-limiting rules, reproducibility requirements, and scope boundaries.

This is a **documentation notebook only**. No scraper code, no parser
implementation, no data downloads, and no Silver or Gold outputs are
produced here.

## Phase 4 Scope

This notebook documents Wikipedia city metadata scraping only.

Wikipedia English city pages are used as the source for contextual city
metadata such as population, area, coordinates, and country context for
the 8 starter cities defined in Phase 2.

**Not in this notebook:**

- Open-Meteo API ingestion (Phase 5).
- EEA historical air quality batch ingestion (Phase 3).
- Kafka producer or consumer logic.
- Spark processing or streaming logic.
- Gold-layer analytics or reporting.

## Source Access Policy

Wikipedia English city pages are accessed via predictable URLs:

```
https://en.wikipedia.org/wiki/<City_Name>
```

For example:

- Vienna: `https://en.wikipedia.org/wiki/Vienna`
- Berlin: `https://en.wikipedia.org/wiki/Berlin`

**Raw HTML storage:**

Fetched raw HTML is stored locally under `data/bronze/wikipedia_html/`.
These files are ignored by Git via the `data/**/*.html` rule in `.gitignore`
and must never be committed to the repository.

**Fetch-once-cache-locally strategy:**

If a local HTML file already exists for a given `city_id`, the scraper
must use the cached file and must not re-fetch from Wikipedia.

## Raw Data Policy

- Raw Wikipedia HTML files are **not committed** to the repository.
- All files under `data/bronze/wikipedia_html/` are **gitignored** via
  `data/**/*.html` in `.gitignore`.
- Raw files must be **reproducible** via documented URLs; every scraped
  page must have its URL, city_id, and fetch date recorded.
- Fetching is **rate-limited**: at least 1 second between requests;
  no burst or parallel requests.
- Only the city pages for the 8 starter cities are in scope; do not
  crawl additional pages or follow internal Wikipedia links.
- Tiny fixed HTML fixtures used only in pytest are acceptable but must
  not be committed.

## Naming Convention

All files stored under `data/bronze/wikipedia_html/` must follow these
naming patterns:

| File type | Pattern | Example |
| --- | --- | --- |
| Production file (one per city, fetched by scraper) | `<city_id>.html` | `vienna_at.html` |
| Sample or test file (HTML excerpt, local validation) | `sample_wikipedia_<city>_<country>.html` | `sample_wikipedia_vienna_at.html` |

Where:

- `<city_id>` is the canonical city identifier from `city_reference.parquet`,
  e.g. `vienna_at`.
- `<city>` is the lower-case city name component, e.g. `vienna`.
- `<country>` is the lower-case two-letter ISO country code, e.g. `at`.

## Rate-Limiting And Politeness Policy

Phase 4 scraping must respect Wikipedia's access guidelines:

| Rule | Requirement |
| --- | --- |
| Minimum delay between requests | At least **1 second** between consecutive page fetches. |
| User-Agent header | Every HTTP request must include a `User-Agent` header identifying the project: `euro-air-quality-pipeline/1.0`. |
| No crawling beyond city pages | Only fetch the approved city page per `city_id`; do not follow internal links or download full site content. |
| Fetch-once strategy | If a local HTML file already exists for a `city_id`, do not re-fetch; use the cached local file. |
| No burst or parallel requests | The scraper must not send concurrent requests. |

Required `User-Agent` header value:

```
User-Agent: euro-air-quality-pipeline/1.0
```

## Git-Ignore Verification

The repository `.gitignore` covers all Wikipedia HTML files through the
following rules:

```gitignore
data/**/*.parquet
data/**/*.csv
data/**/*.json
data/**/*.html
data/**/checkpoints/**
!**/.gitkeep
```

To verify that a local Wikipedia HTML file is correctly ignored before
attempting to add it, run:

```bash
git check-ignore -v data/bronze/wikipedia_html/sample_vienna.html
```

The expected output is a line referencing the `.gitignore` rule and the
file path. If the file does not appear as ignored, check the `.gitignore`
rules before proceeding.

## Reproducibility Contract

Because raw Wikipedia HTML files are not committed, reproducibility depends
on documentation. The following information must be recorded in
`docs/data_sources.md` or in this notebook for every Wikipedia page fetched
in Phase 4:

| Item | Example |
| --- | --- |
| `city_id` | `vienna_at` |
| `wikipedia_url` | `https://en.wikipedia.org/wiki/Vienna` |
| `scraped_at` (UTC) | `2026-05-30T17:00:00Z` |
| Local path | `data/bronze/wikipedia_html/vienna_at.html` |

This table will be populated during Phase 4 implementation issues.

## Phase 4 Scope Boundary

**Included in this issue (4.1):**

- Document the accepted Wikipedia source access path.
- Define raw-data policy and naming conventions.
- Define rate-limiting and politeness policy.
- Confirm git-ignore behaviour.
- Record scope boundary.

**Not included in this issue:**

- Any Wikipedia page download or scraping.
- HTML parser or infobox extraction implementation.
- Silver Parquet city metadata output.
- Any Open-Meteo, EEA, Kafka, or Spark work.

## Output

The Silver output produced by later Phase 4 implementation issues is:

```text
data/silver/city_metadata.parquet
```

This file will contain parsed city metadata fields extracted from Wikipedia
infoboxes, including population, area, coordinates, and country context.

It is **not produced in this issue**. This notebook documents policy only.

The output Parquet file is ignored by Git via `data/**/*.parquet` and must
not be committed.

## Join Contract

All records in `data/silver/city_metadata.parquet` join to
`city_reference.parquet` via the `city_id` field.

Wikipedia-derived values are **contextual metadata only**. They must not
be treated as official statistical ground truth and must not override
canonical coordinates or country codes defined in `city_reference.parquet`.

| Join key | `city_id` (from `city_reference.parquet`) |
| --- | --- |
| Every Wikipedia metadata row must have a non-null `city_id`. | Rows without a matching `city_id` in `city_reference.parquet` must be rejected or flagged. |
| Wikipedia values are supplementary. | They enrich the city reference but do not replace it. |

## Wikipedia City Metadata Schema

This section defines the Silver output schema for Phase 4 Wikipedia city
metadata. The schema is defined before implementation to enable testing
and schema validation. It extends the Phase 2 Wikipedia Metadata Join Rules
with a concrete field-level contract.

| field | type | required | nullability | rule |
| --- | --- | --- | --- | --- |
| `city_id` | string | yes | non-null | Join key from `city_reference.parquet`; must match exactly; must not be a free-text city name. |
| `city_name` | string | yes | non-null | Display name from the Wikipedia page title. |
| `population` | integer | no | nullable | Contextual value; null when not parseable; see `metadata_notes`. |
| `area_km2` | float | no | nullable | Contextual area in km²; null when not parseable; see `metadata_notes`. |
| `population_density` | float | no | nullable | Calculated when `population` and `area_km2` are both non-null; otherwise null. |
| `country` | string | no | nullable | Contextual country name from infobox; null when missing or ambiguous. |
| `wikipedia_url` | string | yes | non-null | Source URL of the scraped Wikipedia page; required for reproducibility. |
| `scraped_at` | string | yes | non-null | ISO 8601 UTC timestamp of the scraping run. |
| `metadata_notes` | string | yes | non-null | Documents parse failures, ambiguous values, or missing fields; `ok` when everything parsed. |
| `source` | string | yes | non-null | Always `wikipedia`. |

## city_id Join Contract

- `city_id` is the sole join key between `city_metadata.parquet` and
  `city_reference.parquet`.
- Every row must have a non-null `city_id` matching an entry in
  `city_reference.parquet`.
- Page titles, URLs, and free-text names are traceability fields only;
  they are never join keys.
- Wikipedia values are contextual and must not override `city_reference.parquet`
  coordinate or country code fields.

## Null And Missing Field Rules

- If a field cannot be parsed, set it to null and add a note in
  `metadata_notes`.
- Never guess or infer values from unrelated pages.
- A city with all nullable fields null is still a valid row if `city_id`,
  `city_name`, `wikipedia_url`, `scraped_at`, `metadata_notes`, and `source`
  are present.
- `population_density` is only computed when both `population` and `area_km2`
  are non-null.

## Silver Output Path

```text
data/silver/city_metadata.parquet
```

Produced by Phase 4 Issue 4.6. Not yet implemented in this issue. Git-ignored.

## Contextual-Only Status

Wikipedia values must not be treated as official statistical ground truth,
analytical baselines, or pipeline dependencies. EEA measurements and
Open-Meteo data must not depend on Wikipedia values being complete.